In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from joblib import Parallel, delayed
from datetime import datetime
import sys
from pathlib import Path
from tqdm import tqdm

import sys
# sys.path.append("../src")
sys.path.append(str(Path.cwd().parent))

# from indicators.indicators import load_symbol, build_indicators
from mdt.mdt import load_symbol
from backtester import engine

import vars

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 40)
pd.set_option('display.max_colwidth', 200)

color_back = "#0d1117"

path_repo = vars.path_repo
path_data = vars.path_data

In [2]:
symbs = vars.symbs

symbs_crypto = [x for x in symbs if "-USD" in x]
symbs_forex = [x for x in symbs if "=X" in x]
symbs_stocks = [x for x in symbs if x not in symbs_crypto + symbs_forex]

print("Total:", len(symbs))
print("Crypto :", len(symbs_crypto), f"({(len(symbs_crypto)/len(symbs))*100:.0f}%)")
print("Forex:", len(symbs_forex), f"({(len(symbs_forex)/len(symbs))*100:.0f}%)")
print("Stocks:", len(symbs_stocks), f"({(len(symbs_stocks)/len(symbs))*100:.0f}%)")

Total: 12
Crypto : 3 (25%)
Forex: 3 (25%)
Stocks: 6 (50%)


In [3]:
symb = "AAPL"
df = load_symbol(symb, "1h", 400) 


# cols_sign = [col for col in df.columns if col.startswith("sign_")] 
# cols_indic = [col for col in df.columns if not col.startswith("sign_")] 

cols_sign = [
# 'sign_random', 'sign_random2', 
# 'sign_macd_rsi_robust', #'sign_macd_rsi_simple', #'sign_macd_rsi_simple_break', #'sign_macd_rsi_simple_bb_pos', 'sign_macd_rsi_bb', 'sign_macd_rsi_bb_ema', 
# 'sign_adx_crec', 'sign_adx_crec_rsi', #'sign_adx_rsi',
# 'sign_adx_rsi_robust', #'sign_adx_rsi_xtrem', 
'sign_vpvma', 'sign_sinergia', #'sign_rsi_bb_rever', #'sign_ma_macd', 'str_squ_kel_bb', 
# 'sign_gold', 
# 'sign_rsi30', #'sign_rsi20', 'sign_rsi15', 'sign_pullb_ema', 'sign_rsi_div', 'sign_supo_resi', 'sign_sobreextema', 'sign_stochastic', 'sign_cmf', 'sign_elderray', 
'sign_willr', #'sign_donchian', 'sign_liquidsweep', 'sign_macd_div1', 'sign_macd_div3', 'sign_macd_div5', 'sign_random3'
]

cols_sign = [
'sign_random', 'sign_random2', 'sign_macd_rsi_robust', 'sign_macd_rsi_simple', 'sign_macd_rsi_simple_break', 'sign_macd_rsi_simple_bb_pos', 'sign_macd_rsi_bb', 'sign_macd_rsi_bb_ema', 
'sign_adx_crec', 'sign_adx_crec_rsi', 'sign_adx_rsi', 'sign_adx_rsi_robust', 'sign_adx_rsi_xtrem', 'sign_vpvma', 'sign_sinergia', 'sign_rsi_bb_rever', 'sign_ma_macd', 'str_squ_kel_bb', 
'sign_gold', 'sign_rsi30', 'sign_rsi20', 'sign_rsi15', 'sign_pullb_ema', 'sign_rsi_div', 'sign_supo_resi', 'sign_sobreextema', 'sign_stochastic', 'sign_cmf', 'sign_elderray', 
'sign_willr', 'sign_donchian', 'sign_liquidsweep', 'sign_macd_div1', 'sign_macd_div3', 'sign_macd_div5', 'sign_random3'
]

cols_indic = [
'Close', 'High', 'Low', 'Open', 'Volume', 'date', 'EMA20', 'EMA50', 'EMA200', 'RSI', 'MACD', 'MACD_signal', 'MACD_hist', 'BB_upper', 'BB_mid', 'BB_lower', 'BB_position', 
'STOCH_K', 'STOCH_D', 'ATR', 'OBV', 'OBV_EMA', 'ADX', 'ADX+DI', 'ADX-DI', 'prev26', 'prev52', 'VPVMA', 'VPVMAS', 'VWAP', 'spread', 'vol_ma', 'volume_ma', 'volume_increasing', 
'order_flow', 'volatility', 'CMF', 'PSAR', 'WillR', 'Donchian_high', 'Donchian_low', 'ema50_distance', 'MACD_bearish_div', 'MACD_bullish_div', 'ind_volume_anomaly', 'ind_compress_range', 
'ind_vwap_deviation', 'ind_3ema', 'ind_adx', 'trend_adx',
]

order = sorted(cols_sign)
# df = df[order]

print("Signals   :", len(cols_sign))
print("Indicators:", len(cols_indic))

Signals   : 36
Indicators: 50


In [4]:
def calculate_score(resume):
    resume = resume.copy()
    variables = ['return', 'sharpe', 'profit_factor', "calmar", "expectancy"]
    cols_rank = []
    for col in variables:
        resume[f'rank_{col}'] = resume[col].fillna(0).rank(method='min', ascending=False).astype(int)
        cols_rank += [f'rank_{col}']

    resume["score"] = 0.30 * resume["rank_sharpe"] + 0.20 * resume["rank_return"] + 0.20 * resume["rank_profit_factor"] + 0.15 * resume["rank_calmar"] + 0.15 * resume["rank_expectancy"]

    resume.drop(columns = cols_rank, inplace = True)


    resume["score"] = np.where(resume["trades"] < 5, resume["score"] * 0.5, resume["score"])
    resume["score"] = np.where(resume["trades_byday"] >= 4, resume["score"] * 0.5, resume["score"])
    resume["score"] = np.where(resume["profit_factor"] < 1.1, resume["score"] * 0.5, resume["score"])

    # if max_drawdown < -0.30:
    #     score *= 0.5

    return resume


def symbol_performance(df, _cols_sign, _symb, atr_sl = 2,
                                                multiplier_tp=1.5,
                                                use_trailing_stop=False,
                                                use_breakeven=False, # Se activa solo si está activo use_trailing_stop
                                                trailing_atr_mult=1,
                                                trailing_activation_r=2,
                                                breakeven_trigger_r=2):

    params = [atr_sl, multiplier_tp, use_trailing_stop, use_breakeven, trailing_atr_mult, trailing_activation_r, breakeven_trigger_r]

    def _symbol(df, signal, _symb, params): 

        resume, trades, equity = engine(df, signal, _symb,
                                        initial_capital=1000,
                                        risk_per_trade=0.01,
                                        fee_rate=0.0007,
                                        slippage_pct=0.0005,
                                        leverage=100,
                                        atr_sl=params[0],
                                        multiplier_tp=params[1], 
                                        use_trailing_stop=params[2],
                                        use_breakeven=params[3],      # Se activa solo si está activo use_trailing_stop
                                        trailing_atr_mult=params[4],
                                        trailing_activation_r=params[5],
                                        breakeven_trigger_r=params[6])
        
        resume["signal"] = signal
        resume["symbol"] = _symb
        trades["signal"] = signal
        trades["symbol"] = _symb
        equity["signal"] = signal
        equity["symbol"] = _symb

        return resume, trades, equity

    results = Parallel(n_jobs=-1, backend="threading")(delayed(_symbol)(df, signal, _symb, params) for signal in _cols_sign)

    resume =  []
    trades = []
    equity = []

    for _ax1, _ax2, _ax3 in results:
        resume.append(_ax1)
        trades.append(_ax2)
        equity.append(_ax3)
        
    resume = pd.concat(resume).sort_values("capital_final", ascending = False)
    trades = pd.concat(trades)
    equity = pd.concat(equity)

    # print(len(resume))

    tmp = list(set(df["date"].astype(str).str[:10].tolist()))
    total = len(tmp)
    resume["trades_byday"] = resume["trades"] / total

    resume["param.atr_sl"] = atr_sl
    resume["param.multiplier_tp"] = multiplier_tp
    resume["param.use_trailing_stop"] = use_trailing_stop
    resume["param.use_breakeven"] = use_breakeven
    resume["param.trailing_atr_mult"] = trailing_atr_mult
    resume["param.trailing_activation_r"] = trailing_activation_r
    resume["param.breakeven_trigger_r"] = breakeven_trigger_r

    resume = calculate_score(resume)
 
    return resume, trades, equity




In [5]:
resume, trades, equity = symbol_performance(df, cols_sign, symb, atr_sl = 2, multiplier_tp = 1.5, use_trailing_stop = False, use_breakeven = False, trailing_atr_mult = 1,
                                                                 trailing_activation_r = 2.5, breakeven_trigger_r = 3)

In [6]:
resume[['symbol', 'signal', "score", 'capital_final', 'pnl', 'return', 'trades', 'win_rate', 'trades_byday', 'expectancy', 'profit_factor', 'sharpe',
        'n_tp', 'n_sl', 'n_ts', 'n_rev', 'fees']] # 'avg_win', 'avg_loss', 'max_drawdown', 

,symbol,signal,score,capital_final,pnl,return,trades,win_rate,trades_byday,expectancy,profit_factor,sharpe,n_tp,n_sl,n_ts,n_rev,fees
0,AAPL,sign_sobreextema,2.0500,1102.677288,102.677288,0.102677,59,0.508475,0.161644,2.188187,1.362748,1.835452,28,29,0,2,52.849934
0,AAPL,sign_rsi20,3.1000,1045.731433,45.731433,0.045731,16,0.562500,0.043836,3.297472,1.688303,1.572784,9,7,0,0,14.130384
0,AAPL,sign_macd_rsi_simple_bb_pos,1.4000,1016.170004,16.170004,0.016170,3,0.666667,0.008219,5.850925,2.614099,1.694438,2,1,0,0,2.752295
0,AAPL,sign_adx_crec_rsi,1.0250,1013.403863,13.403863,0.013404,1,1.000000,0.002740,13.998437,inf,1.803798,1,0,0,0,1.178941
0,AAPL,sign_rsi15,2.5000,1004.582078,4.582078,0.004582,4,0.500000,0.010959,1.614661,1.292243,0.301972,2,2,0,0,3.788464
0,AAPL,sign_macd_rsi_robust,6.0000,1003.024986,3.024986,0.003025,13,0.461538,0.035616,0.694099,1.116508,0.158280,6,7,0,0,11.998273
0,AAPL,sign_macd_rsi_bb_ema,3.6750,993.672916,-6.327084,-0.006327,7,0.428571,0.019178,-0.343021,0.945249,-0.290747,3,4,0,0,7.833038
0,AAPL,sign_adx_rsi,5.4750,992.462409,-7.537591,-0.007538,5,0.400000,0.013699,-0.975346,0.852780,-0.426821,2,3,0,0,5.314323
0,AAPL,sign_adx_rsi_xtrem,6.3500,988.960699,-11.039301,-0.011039,1,0.000000,0.002740,-10.651910,0.000000,-2.959407,0,1,0,0,0.767593
0,AAPL,sign_adx_rsi_robust,6.3500,988.960699,-11.039301,-0.011039,1,0.000000,0.002740,-10.651910,0.000000,-2.959407,0,1,0,0,0.767593


### Tuning

In [7]:
# Calculate train and test
symb = "AAPL"
df = load_symbol(symb, "1h", 600) 

In [8]:
porc_train = 0.7 
porc_eval = 0.15

df["mes"] = df["date"].dt.strftime("%Y-%m-%d")
ax = df.drop_duplicates("mes")
c = int(len(ax)*porc_train)
c2 = int(len(ax)*(porc_eval+porc_train))

dates_train = [ax.iloc[:c]["mes"].tolist()[0], ax.iloc[:c]["mes"].tolist()[-1]]
dates_eval = [ax.iloc[c:]["mes"].tolist()[0], ax.iloc[:c2]["mes"].tolist()[-1]]
dates_oos = [ax.iloc[c2:]["mes"].tolist()[0], ax.iloc[c2:]["mes"].tolist()[-1]]

df_train = df[(df["date"] >= dates_train[0]) & (df["date"] <= dates_train[1])]
df_eval = df[(df["date"] >= dates_eval[0]) & (df["date"] <= dates_eval[1])]
df_oos = df[(df["date"] >= dates_oos[0]) & (df["date"] <= dates_oos[1])]

print("Train:", dates_train)
print("Eval :", dates_eval)
print("OOS  :", dates_oos, "\n")

print(f"{len(df_train)/len(df)*100:.1f}%")
print(f"{len(df_eval)/len(df)*100:.1f}%")
print(f"{len(df_oos)/len(df)*100:.1f}%")

Train: ['2024-07-15', '2026-02-06']
Eval : ['2026-02-09', '2026-06-10']
OOS  : ['2026-06-11', '2026-10-12'] 

69.5%
15.0%
15.0%


In [9]:
"""
6. Qué optimizaría primero en tu caso No optimizaría todos los parámetros de una sola vez. Lo haría en tres etapas:
1. SL y TP: atr_sl, multiplier_tp,
2. Gestión de beneficios: trailing stop, multiplicador ATR y activación.
3. Breakeven: activación y compatibilidad con trailing.


Parámetro	Valores iniciales
atr_sl	1.0, 1.5, 2.0, 2.5, 3.0
multiplier_tp	1.0, 1.5, 2.0, 2.5, 3.0
use_trailing_stop	False, True
trailing_atr_mult	1.0, 1.5, 2.0
trailing_activation_r	1.0, 1.5, 2.0, 2.5, 3.0
use_breakeven	False, True
breakeven_trigger_r	1.0, 1.5, 2.0, 2.5, 3.0
"""

'\n6. Qué optimizaría primero en tu caso No optimizaría todos los parámetros de una sola vez. Lo haría en tres etapas:\n1. SL y TP: atr_sl, multiplier_tp,\n2. Gestión de beneficios: trailing stop, multiplicador ATR y activación.\n3. Breakeven: activación y compatibilidad con trailing.\n\n\nParámetro\tValores iniciales\natr_sl\t1.0, 1.5, 2.0, 2.5, 3.0\nmultiplier_tp\t1.0, 1.5, 2.0, 2.5, 3.0\nuse_trailing_stop\tFalse, True\ntrailing_atr_mult\t1.0, 1.5, 2.0\ntrailing_activation_r\t1.0, 1.5, 2.0, 2.5, 3.0\nuse_breakeven\tFalse, True\nbreakeven_trigger_r\t1.0, 1.5, 2.0, 2.5, 3.0\n'

In [10]:
def build_param_grid():
    grid = []
    for atr_sl in [1, 1.5, 2, 2.5]:
        for tp in [1, 1.5, 2, 2.5]:
            for use_trailing_stop in [True, False]:
                for use_breakeven in [True, False]:
                    for trailing_atr_mult in [1]:
                        for trailing_activation_r in [2]:
                            for breakeven_trigger_r in [2]:
                                
                                if use_breakeven and not use_trailing_stop: # Evita activar dos mecanismos de gestión en configuraciones que todavía no quieres explorar.
                                    continue

                                # if not use_trailing_stop:
                                #     trailing_atr_mult = 0
                                #     trailing_activation_r = 0
                                #     breakeven_trigger_r = 0

                                else:
                                    params = {
                                        "atr_sl": atr_sl,
                                        "multiplier_tp": tp,
                                        "use_trailing_stop": use_trailing_stop,
                                        "use_breakeven": use_breakeven,
                                        "trailing_atr_mult": trailing_atr_mult,
                                        "trailing_activation_r": trailing_activation_r,
                                        "breakeven_trigger_r": breakeven_trigger_r 
                                    }

                                    grid.append(params)
    return grid

param_grid = build_param_grid()
param_grid = pd.DataFrame(param_grid).drop_duplicates()

print("Setups:", len(param_grid))
param_grid.head(10)

Setups: 48


,atr_sl,multiplier_tp,use_trailing_stop,use_breakeven,trailing_atr_mult,trailing_activation_r,breakeven_trigger_r
0,1.0,1.0,True,True,1,2,2
1,1.0,1.0,True,False,1,2,2
2,1.0,1.0,False,False,1,2,2
3,1.0,1.5,True,True,1,2,2
4,1.0,1.5,True,False,1,2,2
5,1.0,1.5,False,False,1,2,2
6,1.0,2.0,True,True,1,2,2
7,1.0,2.0,True,False,1,2,2
8,1.0,2.0,False,False,1,2,2
9,1.0,2.5,True,True,1,2,2


In [11]:
def tunner_sl_tp(df, cols_sign, symb, param_grid):
    df = df.copy()
    resume_tun = []
    chunk_size = 50
    for inicio in tqdm(range(0, len(param_grid), chunk_size)):
        chunk = param_grid.iloc[inicio:inicio + chunk_size]
        resultados = Parallel(n_jobs=-1, backend="loky")(
            delayed(symbol_performance)(
                df,
                cols_sign,
                symb,
                atr_sl=row["atr_sl"],
                multiplier_tp=row["multiplier_tp"],
                use_trailing_stop=row["use_trailing_stop"],
                use_breakeven=row["use_breakeven"],
                trailing_atr_mult=row["trailing_atr_mult"],
                trailing_activation_r=row["trailing_activation_r"],
                breakeven_trigger_r=row["breakeven_trigger_r"]
            )
            for _, row in chunk.iterrows())
        resume_tun.extend([resultado[0] for resultado in resultados])


    # for i in tqdm(range(0, len(param_grid))):
    #     row = param_grid.iloc[i]
    #     _resume, _trades, _equity = symbol_performance(
    #                                                 df,
    #                                                 cols_sign,
    #                                                 symb,
    #                                                 atr_sl=row["atr_sl"],
    #                                                 multiplier_tp=row["multiplier_tp"],
    #                                                 use_trailing_stop=row["use_trailing_stop"],
    #                                                 use_breakeven=row["use_breakeven"],
    #                                                 trailing_atr_mult=row["trailing_atr_mult"],
    #                                                 trailing_activation_r=row["trailing_activation_r"],
    #                                                 breakeven_trigger_r=row["breakeven_trigger_r"])
    #     resume_tun.append(_resume)
        

    resume_tun = pd.concat(resume_tun, ignore_index=True)
    resume_tun.drop_duplicates(['symbol', 'signal', 'capital_final', 'pnl', 'return', 'trades', 'win_rate', 'expectancy', 'avg_return', 'max_drawdown', 'sharpe'], inplace = True)
    resume_tun = calculate_score(resume_tun)
    resume_tun = resume_tun[["score"] + [col for col in resume_tun.columns if col != "score"]].sort_values("score", ascending=True)
    return resume_tun

In [12]:
k_top = 3
cols_del = ["capital", "avg_win", "avg_loss", "median_return", "sortino","cagr", "calmar", "avg_trade", "profit_factor", "avg_bars",
            "max_loss_streak", "max_win_streak", "recovery_factor", "var_95", "cvar_95", "exposure", "open_trade"]

In [13]:
# 1. Tuning in train
# resume_train = tunner_sl_tp(df_train, cols_sign, symb, param_grid)

resume_train = []
for _sign in cols_sign:
    print(_sign)
    _ = tunner_sl_tp(df_oos, [_sign], symb, param_grid)
    resume_train.append(_)
resume_train = pd.concat(resume_train)

tmp = []
for _sign in cols_sign:
    _ = resume_train[resume_train["signal"] == _sign].sort_values(["signal", "score"], ascending=True).head(k_top)
    tmp.append(_)
resume_train = pd.concat(tmp)
resume_train["dataset"] = "train"
resume_train.drop(columns=cols_del).head(15)

sign_random


100%|██████████| 1/1 [00:10<00:00, 10.02s/it]


sign_random2


100%|██████████| 1/1 [00:04<00:00,  4.02s/it]


sign_macd_rsi_robust


100%|██████████| 1/1 [00:03<00:00,  3.99s/it]


... (63 outputs left out)

100%|██████████| 1/1 [00:04<00:00,  4.10s/it]


sign_random3


100%|██████████| 1/1 [00:04<00:00,  4.25s/it]


,score,symbol,signal,capital_final,pnl,return,trades,win_rate,expectancy,avg_return,max_drawdown,sharpe,n_tp,n_sl,n_ts,n_rev,fees,trades_byday,param.atr_sl,param.multiplier_tp,param.use_trailing_stop,param.use_breakeven,param.trailing_atr_mult,param.trailing_activation_r,param.breakeven_trigger_r,dataset
33,1.000,AAPL,sign_random,993.988720,-6.011280,-0.006011,33,0.454545,0.257792,0.046122,-0.086613,-0.099479,5,12,0,16,29.037068,0.392857,2.0,2.5,True,True,1,2,2,train
45,1.000,AAPL,sign_random,1016.266632,16.266632,0.016267,31,0.483871,0.882384,0.137937,-0.063997,0.995261,5,9,0,17,22.207271,0.369048,2.5,2.5,True,True,1,2,2,train
42,1.500,AAPL,sign_random,991.722376,-8.277624,-0.008278,31,0.483871,0.086452,-0.005830,-0.068973,-0.310706,5,9,0,17,21.930745,0.369048,2.5,2.0,True,True,1,2,2,train
47,1.500,AAPL,sign_random2,1057.928772,57.928772,0.057929,16,0.500000,3.691130,0.955654,-0.058152,3.588544,4,5,0,7,11.883910,0.190476,2.5,2.5,False,False,1,2,2,train
36,2.150,AAPL,sign_random2,1042.942180,42.942180,0.042942,21,0.619048,2.412792,0.477325,-0.049595,3.777948,11,4,0,6,15.479762,0.250000,2.5,1.0,True,True,1,2,2,train
45,2.500,AAPL,sign_random2,1055.238175,55.238175,0.055238,16,0.500000,3.523055,0.913588,-0.058152,3.433652,4,6,0,6,11.860363,0.190476,2.5,2.5,True,True,1,2,2,train
12,0.600,AAPL,sign_macd_rsi_robust,1008.529139,8.529139,0.008529,1,1.000000,9.075357,1.163043,-0.007529,2.878600,1,0,0,0,1.085705,0.011905,1.5,1.0,True,True,1,2,2,train
9,0.900,AAPL,sign_macd_rsi_robust,1010.234764,10.234764,0.010235,2,0.500000,5.941935,0.495118,-0.012236,2.284985,1,1,0,0,3.288743,0.023810,1.0,2.5,True,True,1,2,2,train
0,1.250,AAPL,sign_macd_rsi_robust,995.400524,-4.599476,-0.004599,2,0.500000,-1.475186,-0.130589,-0.012236,-1.361878,1,1,0,0,3.299120,0.023810,1.0,1.0,True,True,1,2,2,train
12,0.600,AAPL,sign_macd_rsi_simple,1008.529139,8.529139,0.008529,1,1.000000,9.075357,1.163043,-0.007529,2.878600,1,0,0,0,1.085705,0.011905,1.5,1.0,True,True,1,2,2,train


In [14]:
param_grid_eval = resume_train[["signal"] + [x for x in resume_train if "param" in x]]
param_grid_eval.columns = [x.replace("param.","") for x in param_grid_eval]
param_grid_eval

,signal,atr_sl,multiplier_tp,use_trailing_stop,use_breakeven,trailing_atr_mult,trailing_activation_r,breakeven_trigger_r
33,sign_random,2.0,2.5,True,True,1,2,2
45,sign_random,2.5,2.5,True,True,1,2,2
42,sign_random,2.5,2.0,True,True,1,2,2
47,sign_random2,2.5,2.5,False,False,1,2,2
36,sign_random2,2.5,1.0,True,True,1,2,2
...,...,...,...,...,...,...,...,...
9,sign_macd_div5,1.0,2.5,True,True,1,2,2
6,sign_macd_div5,1.0,2.0,True,True,1,2,2
45,sign_random3,2.5,2.5,True,True,1,2,2
47,sign_random3,2.5,2.5,False,False,1,2,2


In [ ]:
# 2. Testing in evaluation
resume_eval = tunner_sl_tp(df_eval, cols_sign, symb, param_grid_eval)
resume_eval["dataset"] = "eval"
resume_eval.drop(columns=cols_del).head(15)

In [ ]:
# 3. Testing in OOS
resume_oos = tunner_sl_tp(df_oos, cols_sign, symb, param_grid_eval)
resume_oos["dataset"] = "oos"
resume_oos.drop(columns=cols_del).head(15)

In [ ]:
resume_glob = pd.concat([resume_train, resume_eval, resume_oos])
cols_param = ['param.atr_sl', 'param.multiplier_tp', 'param.use_trailing_stop', 'param.use_breakeven', 'param.trailing_atr_mult', 'param.trailing_activation_r', 'param.breakeven_trigger_r']
resume_glob["key"] = resume_glob[cols_param].astype(str).agg("".join, axis = 1)
resume_glob = resume_glob[["dataset", "key"] + [col for col in resume_glob.columns if col not in ["dataset", "key"]]].sort_values(cols_param, ascending=True)
resume_glob.drop(columns=cols_del).head(15)

In [ ]:
ph = vars.path_data + f"{symb}/"
resume_glob.to_parquet(ph)

In [ ]:
qwe

## Testing

In [16]:
# 2. Testing in evaluation
resume_eval = []
for _sign in cols_sign:
    _params = param_grid_eval[param_grid_eval["signal"] == _sign]
    _ = tunner_sl_tp(df_eval, [_sign], symb, _params)
    resume_eval.append(_)
resume_eval = pd.concat(resume_eval)
resume_eval["dataset"] = "eval"
resume_eval.drop(columns=cols_del).head(15)

100%|██████████| 1/1 [00:00<00:00,  2.66it/s]


,score,symbol,signal,capital_final,pnl,return,trades,win_rate,expectancy,avg_return,max_drawdown,sharpe,n_tp,n_sl,n_ts,n_rev,fees,trades_byday,param.atr_sl,param.multiplier_tp,param.use_trailing_stop,param.use_breakeven,param.trailing_atr_mult,param.trailing_activation_r,param.breakeven_trigger_r,dataset
1,0.650,AAPL,sign_random,844.124248,-155.875752,-0.155876,43,0.232558,-3.201397,-0.605018,-0.176823,-11.367403,1,18,0,24,31.834266,0.511905,2.5,2.5,True,True,1,2,2,eval
2,1.000,AAPL,sign_random,843.487919,-156.512081,-0.156512,43,0.232558,-3.217840,-0.616279,-0.178899,-11.578233,2,17,0,24,31.695019,0.511905,2.5,2.0,True,True,1,2,2,eval
0,1.350,AAPL,sign_random,798.394371,-201.605629,-0.201606,46,0.217391,-3.883378,-0.651012,-0.224648,-12.596776,1,25,0,20,40.509315,0.547619,2.0,2.5,True,True,1,2,2,eval
1,0.650,AAPL,sign_random2,940.281000,-59.719000,-0.059719,32,0.531250,-1.473334,-0.264227,-0.083062,-5.205730,7,11,0,14,25.181391,0.380952,2.5,1.0,True,True,1,2,2,eval
0,1.125,AAPL,sign_random2,919.485999,-80.514001,-0.080514,30,0.466667,-2.291180,-0.393293,-0.085968,-5.087791,1,11,0,18,23.598403,0.357143,2.5,2.5,False,False,1,2,2,eval
2,1.225,AAPL,sign_random2,921.588217,-78.411783,-0.078412,30,0.466667,-2.220900,-0.391909,-0.083878,-5.595430,0,13,0,17,23.600339,0.357143,2.5,2.5,True,True,1,2,2,eval
0,0.500,AAPL,sign_macd_rsi_robust,1000.000000,0.000000,0.000000,0,0.000000,NaN,NaN,0.000000,NaN,0,0,0,0,0.000000,0.000000,1.5,1.0,True,True,1,2,2,eval
1,0.500,AAPL,sign_macd_rsi_simple,996.289728,-3.710272,-0.003710,3,0.333333,-0.228940,-0.071983,-0.039272,-0.500132,1,2,0,0,6.045269,0.035714,1.0,2.5,True,True,1,2,2,eval
2,0.500,AAPL,sign_macd_rsi_simple,1001.551176,1.551176,0.001551,3,0.666667,1.539683,0.169069,-0.020729,0.354035,2,1,0,0,6.130368,0.035714,1.0,1.0,True,True,1,2,2,eval
0,0.750,AAPL,sign_macd_rsi_simple,984.391457,-15.608543,-0.015609,3,0.333333,-4.530351,-0.528453,-0.032862,-3.589239,1,2,0,0,4.043078,0.035714,1.5,1.0,True,True,1,2,2,eval


In [17]:
# 3. Testing in OOS
resume_oos = []
for _sign in cols_sign:
    _params = param_grid_eval[param_grid_eval["signal"] == _sign]
    _ = tunner_sl_tp(df_oos, [_sign], symb, _params)
    resume_oos.append(_)
resume_oos = pd.concat(resume_oos)
resume_oos["dataset"] = "eval"
resume_oos.drop(columns=cols_del).head(15)

100%|██████████| 1/1 [00:00<00:00,  2.77it/s]


,score,symbol,signal,capital_final,pnl,return,trades,win_rate,expectancy,avg_return,max_drawdown,sharpe,n_tp,n_sl,n_ts,n_rev,fees,trades_byday,param.atr_sl,param.multiplier_tp,param.use_trailing_stop,param.use_breakeven,param.trailing_atr_mult,param.trailing_activation_r,param.breakeven_trigger_r,dataset
0,1.0000,AAPL,sign_random,993.988720,-6.011280,-0.006011,33,0.454545,0.257792,0.046122,-0.086613,-0.099479,5,12,0,16,29.037068,0.392857,2.0,2.5,True,True,1,2,2,eval
1,1.0000,AAPL,sign_random,1016.266632,16.266632,0.016267,31,0.483871,0.882384,0.137937,-0.063997,0.995261,5,9,0,17,22.207271,0.369048,2.5,2.5,True,True,1,2,2,eval
2,1.5000,AAPL,sign_random,991.722376,-8.277624,-0.008278,31,0.483871,0.086452,-0.005830,-0.068973,-0.310706,5,9,0,17,21.930745,0.369048,2.5,2.0,True,True,1,2,2,eval
0,1.5000,AAPL,sign_random2,1057.928772,57.928772,0.057929,16,0.500000,3.691130,0.955654,-0.058152,3.588544,4,5,0,7,11.883910,0.190476,2.5,2.5,False,False,1,2,2,eval
1,2.0000,AAPL,sign_random2,1042.942180,42.942180,0.042942,21,0.619048,2.412792,0.477325,-0.049595,3.777948,11,4,0,6,15.479762,0.250000,2.5,1.0,True,True,1,2,2,eval
2,2.5000,AAPL,sign_random2,1055.238175,55.238175,0.055238,16,0.500000,3.523055,0.913588,-0.058152,3.433652,4,6,0,6,11.860363,0.190476,2.5,2.5,True,True,1,2,2,eval
0,0.6000,AAPL,sign_macd_rsi_robust,1008.529139,8.529139,0.008529,1,1.000000,9.075357,1.163043,-0.007529,2.878600,1,0,0,0,1.085705,0.011905,1.5,1.0,True,True,1,2,2,eval
2,0.7500,AAPL,sign_macd_rsi_robust,995.400524,-4.599476,-0.004599,2,0.500000,-1.475186,-0.130589,-0.012236,-1.361878,1,1,0,0,3.299120,0.023810,1.0,1.0,True,True,1,2,2,eval
1,0.9000,AAPL,sign_macd_rsi_robust,1010.234764,10.234764,0.010235,2,0.500000,5.941935,0.495118,-0.012236,2.284985,1,1,0,0,3.288743,0.023810,1.0,2.5,True,True,1,2,2,eval
0,0.6000,AAPL,sign_macd_rsi_simple,1008.529139,8.529139,0.008529,1,1.000000,9.075357,1.163043,-0.007529,2.878600,1,0,0,0,1.085705,0.011905,1.5,1.0,True,True,1,2,2,eval


In [18]:
grafico

NameError: name 'grafico' is not defined

In [ ]:
qwe

## Graph

In [ ]:
if "reason" in trades.columns:

    exit_analysis = (
        trades.groupby("reason")
        .agg(
            operations=("return", "size"),
            total_pnl=("return", "sum"),
            avg_pnl=("return", "mean"),
            win_rate=("return", lambda x: (x > 0).mean()),
            median_pnl=("return", "median")
        )
        .sort_values("total_pnl", ascending=False)
    )

    display(exit_analysis)

    exit_analysis["total_pnl"].plot(
        kind="bar",
        figsize=(12, 4)
    )

    plt.title("PnL acumulado por motivo de salida")
    plt.xlabel("Motivo de salida")
    plt.ylabel("PnL total")
    plt.xticks(rotation=45)
    plt.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
import matplotlib.pyplot as plt


dfpl = equity[equity["signal"] == "sign_rsi30"]
dfpl = dfpl.sort_values("time").copy()

dfpl["equity_peak"] = dfpl["equity"].cummax()

dfpl["drawdown"] = (
    dfpl["equity"] / dfpl["equity_peak"] - 1
)

fig, ax = plt.subplots(figsize=(14, 3))

ax.plot(dfpl["time"], dfpl["equity"])

ax.set_title("Equity Curve")
ax.set_xlabel("Fecha")
ax.set_ylabel("Patrimonio ($)")
ax.grid(True)
ax.axhline(1000, linestyle = "--", color = "red")

plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(14, 3))

ax.plot(dfpl["time"], dfpl["drawdown"] * 100)

ax.set_title("Drawdown")
ax.set_xlabel("Fecha")
ax.set_ylabel("Drawdown (%)")
ax.grid(True)
ax.axhline(0, linestyle = "--", color = "red")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt


dfpl = equity
dfpl = dfpl.sort_values("time").copy()

dfpl["equity_peak"] = dfpl["equity"].cummax()

dfpl["drawdown"] = (
    dfpl["equity"] / dfpl["equity_peak"] - 1
)

fig, ax = plt.subplots(figsize=(14, 6.5))

for sign in cols_sign:
    tmp = dfpl[dfpl["signal"] == sign]
    ax.plot(tmp["time"], tmp["equity"], label = sign)

ax.set_title("Equity Curve")
ax.set_xlabel("Fecha")
ax.set_ylabel("Patrimonio ($)")
ax.grid(True)
ax.axhline(1000, linestyle = "--", color = "red")

plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
qewe